# Agència Catalana de l'Aigua
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 22-06-2026<br>

**Introduction:**<br>
This code preprocesses the data regarding gauging stations and reservoirs downloaded from [Agència Catalana de l'Aigua](https://analisi.transparenciacatalunya.cat/es/Medi-Ambient/Xarxes-de-control-del-medi-consulta-de-l-aigua-i-e/wc95-u57z/about_data). 

The **station data** are a set of Excel files that include stations metadata (name, coordinates, river) and the time series of river discharge, total discharge or reservoir outflow. This data is cleaned and reorganize to generate a shapefile of gauging stations and a set of Parquet files with the daily discharge time series for each station. 

The **reservoir data** are a set of Excel files that include reservoir metadata (name, coordinates) and the time series of storage, level and filling. This data is cleaned and reorganize to generate a shapefile of gauging stations and a set of Parquet files with the daily time series for each reservoir. The outflow time series that was included in the station data is be concatenated to the reservoir time series.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import yaml

from ocab.aca import get_station_data, get_reservoir_data

## Configuration

In [2]:
# path where the data is stored
path_aca = Path('/home/casadoj/Data/ACA')

# paths where results will be saved
path_results = path_aca / 'processed'
path_gis = path_results / 'GIS'
path_ts = path_results / 'timeseries'
for path in [path_gis, path_ts]:
    path.mkdir(parents=True, exist_ok=True)

# map ACA reservoirs with other datasets
file_map_datasets = Path('map_reservoirs_ACA.yml')

## Import Data

### Gauging stations

Here I load and prepocess the raw Excel files downloaded from the ACA website. The result are two objects: `timeseries` is a dictionary that contains the time series for each station, and `stations` is a DataFrame with the attributes.

In [3]:
kind = 'stations'

# time series will be saved in a dictionary a
# attributes in a pandas.DataFrame
discharge = {}
stations = pd.DataFrame()

# read raw Excel files iteratively
path_in = path_aca / 'raw' / kind
files = sorted(list(path_in.glob(f'{kind}_*.xlsx')))
last_year = int(files[-1].stem.split('_')[-1])
for file in tqdm(files, desc='files'):

    # get attributes and time series
    attrs, ts = get_station_data(file)

    # update attributes
    stations = pd.concat([stations, attrs], axis=0).drop_duplicates()

    # update time series
    for ID in ts:
        if ID in discharge:
            discharge[ID] = pd.concat(
                (discharge[ID], ts[ID]),
                axis=0
            ).asfreq('D').sort_index(axis=0)
        else:
            discharge[ID] = ts[ID]
stations.sort_index(inplace=True)
stations.index.name = 'id_saih'

print(f'No. stations:\t\t{len(stations)}')
print(f'No. time series:\t{len(discharge)}')

files:   0%|          | 0/8 [00:00<?, ?it/s]

No. stations:		106
No. time series:	106


In [4]:
# export raw stations
stations.to_file(path_gis / 'stations_aca_raw.geojson', driver='GeoJSON')

#### Pre-process

In [5]:
# remove reservoirs and some specific IDs
remove = ['EA085_C4085_EA120', '17117-0159']
stn_ids = [ID for ID in stations.index if ID.startswith('EA') and (ID not in remove)]
stations = stations.loc[stn_ids]

# select variable in case of multiple
rename_cols = {'total_discharge': 'discharge'}
for ID, ts in discharge.items():
    if ts.shape[1] > 1:
        if 'total_discharge' in ts.columns:
            discharge[ID] = ts[['total_discharge']].rename(columns=rename_cols)
        elif 'discharge' in ts.columns:
            discharge[ID] = ts[['discharge']]

# combine duplicates (keep streamflow+channel with simple ID)
duplicates = {
    'EA066': 'EA066_C4066',
    'EA078': 'EA078_C4078',
}
for ID_keep, ID_remove in duplicates.items():
    if ID_remove in discharge:
        discharge[ID_keep] = pd.DataFrame({
            'discharge': discharge[ID_remove]['total_discharge'].combine_first(discharge[ID_keep]['discharge'])
        })
        del discharge[ID_remove]
    if ID_remove in stations.index:
        stations.drop(index=ID_remove, inplace=True)

# rename 'total_discharge' as 'discharge'
for ID, ts in discharge.items():
    discharge[ID] = ts.rename(columns=rename_cols)

# remove duplicate codes from station IDs
stations.index = [ID.split('_')[0] for ID in stations.index]
discharge = {ID.split('_')[0]: ts for ID, ts in discharge.items()}

print(f'No. stations:\t\t{len(stations)}')
print(f'No. time series:\t{len(discharge)}')


# add catchment area
area = pd.read_csv('stations.csv', index_col='id_saih')
stations.loc[area.index, area.columns] = area

# add basin
stations['basin'] = 'CATALUÑA'

# add start, end and current status
for ID in stations.index:
    ts = discharge[ID]
    start, end = ts.index.min(), ts.index.max()
    stations.loc[ID, 'start'] = start.year
    stations.loc[ID, 'end'] = end.year if end.year < last_year else np.nan
    stations.loc[ID, 'active'] = 1 if end.year == last_year else 0
stations[['start', 'end', 'active']] = stations[['start', 'end', 'active']].astype('Int64')

No. stations:		92
No. time series:	103


### Reservoirs

In [6]:
kind = 'reservoirs'

# time series will be saved in a dictionary and reservoir attributes in a pandas.DataFrame
resops = {}
reservoirs = pd.DataFrame()

# read raw Excel files iteratively
path_in = path_aca / 'raw' / kind
files = sorted(list(path_in.glob(f'{kind}_*.xlsx')))
last_year = int(files[-1].stem.split('_')[-1])
for file in tqdm(files, desc='files'):

    # get attributes and time series
    attrs, ts = get_reservoir_data(file)

    # update attributes
    reservoirs = pd.concat([reservoirs, attrs], axis=0).drop_duplicates()

    # update time series
    for ID in ts:
        if ID in resops:
            resops[ID] = pd.concat(
                (resops[ID], ts[ID]),
                axis=0
            ).asfreq('D').sort_index(axis=0)
        else:
            resops[ID] = ts[ID]
reservoirs.sort_index(inplace=True)
reservoirs.index.name = 'id_saih'

print(f'No. reservoirs:\t\t{len(reservoirs)}')
print(f'No. time series:\t{len(resops)}')

# export raw reservoirs
reservoirs.to_file(path_gis / 'reservoirs_aca_raw.geojson', driver='GeoJSON')

files:   0%|          | 0/8 [00:00<?, ?it/s]

No. reservoirs:		13
No. time series:	13


#### Pre-process

##### Attributes

In [7]:
# add catchment area and reservoir capacity
attrs = pd.read_csv('reservoirs.csv', index_col='id_saih')
reservoirs.loc[attrs.index, attrs.columns] = attrs

In [8]:
# load mapping between reservoir datasets
if file_map_datasets.is_file():
    # read mapping
    with open(file_map_datasets, 'r') as file:
        map_datasets = yaml.safe_load(file)

    # add codes to the attributes
    keys = list(next(iter(map_datasets.values())).keys())
    cols = {key: f'id_{key.lower()}' for key in keys}
    for key, col in cols.items():
        reservoirs[col] = reservoirs.index.map({
            ID: dct[key] for ID, dct in map_datasets.items()
            })
    reservoirs[list(cols.values())] = reservoirs[list(cols.values())].astype('Int64')

In [9]:
# add basin
reservoirs['basin'] = 'CATALUÑA'

# add start, end and current status
for ID in reservoirs.index:
    ts = resops[ID]
    start, end = ts.index.min(), ts.index.max()
    reservoirs.loc[ID, 'start'] = start.year
    reservoirs.loc[ID, 'end'] = end.year if end.year < last_year else np.nan
    reservoirs.loc[ID, 'active'] = 1 if end.year == last_year else 0
reservoirs[['start', 'end', 'active']] = reservoirs[['start', 'end', 'active']].astype('Int64')

##### Time series

In [10]:
# combine reservoir and outflow time series
for ID in reservoirs.index:
    if ID in discharge:
        resops[ID] = pd.concat([resops[ID], discharge[ID]], axis=1, sort=True)
        del discharge[ID]

## Export

### Stations
#### Attributes

In [11]:
# add ID from 10000 onward
root_ID = 10000
stations['id'] = root_ID + np.arange(1, len(stations) + 1)
stations.index.name = 'id_saih'

# export
stations.to_file(path_gis / 'stations_aca.geojson', driver='GeoJSON')

#### Time series

In [12]:
path = path_ts / 'stations'
path.mkdir(exist_ok=True)
for ID in stations.index:
    discharge[ID].to_parquet(path / f'{ID}.parquet')

### Reservoirs
#### Attributes

In [13]:
# add ID from 10100 onward
root_ID = 10100
reservoirs['id'] = [root_ID + int(ID.strip('E')) for ID in reservoirs.index]
reservoirs.index.name = 'id_saih'

# export
reservoirs.to_file(path_gis / 'dams_aca.geojson', driver='GeoJSON')

#### Time series

In [14]:
path = path_ts / 'reservoirs'
path.mkdir(exist_ok=True)
for ID in reservoirs.index:
    resops[ID].to_parquet(path / f'{ID}.parquet')